# Quadratic Equation

## Conditional Workflow using LangGraph

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal

In [ ]:
# Define state

class AgentState(TypedDict):
    a: int
    b: int
    c: int
    
    equation: str
    discriminant: float
    result: str

In [ ]:
# Define model
def show_equation(state: AgentState):
    equation = f"{state['a']} ^ 2 + {state['b']} + {state['c']}"
    return {"equation": equation}

def calculate_discriminant(state: AgentState):
    discriminant = state['b']**2 - (4 * state['a'] * state['c'])
    return {"discriminant": discriminant}

def no_real_roots(state: AgentState):
    result = f"No real roots"
    return {"result": result}

def real_roots(state: AgentState):
    root1 = (-state['b']) + (state['discriminant'] ** 0.5) / (2 * state['a'])
    root2 = (-state['b']) - (state['discriminant'] ** 0.5) / (2 * state['a'])
    result = f"The roots are {root1} and {root2}"
    return {"result": result}

def repeated_roots(state: AgentState):
    root = (-state['b']) / (2 * state['a'])
    result = f"Only repeating roots is {root}"
    return {"result": result}


def check_condition(state: AgentState) -> Literal["real_roots", "repeated_roots", "no_real_roots"]:
    if state['discriminant'] > 0:
        return "real_roots"
    elif state['discriminant'] == 0:
        return "repeated_roots"
    return "no_real_roots"

In [ ]:
# define graph

graph = StateGraph(AgentState)

# Add nodes
graph.add_node("show_equation", show_equation)
graph.add_node("calculate_discriminant", calculate_discriminant)
graph.add_node("no_real_roots", no_real_roots)
graph.add_node("real_roots", real_roots)
graph.add_node("repeated_roots", repeated_roots)

# Add edges
graph.add_edge(START, "show_equation")
graph.add_edge("show_equation", "calculate_discriminant")

graph.add_conditional_edges("calculate_discriminant", check_condition)

graph.add_edge("no_real_roots", END)
graph.add_edge("real_roots", END)
graph.add_edge("repeated_roots", END)

# compile graph
workflow = graph.compile()

In [ ]:
# visualize graph
from IPython.display import Image
Image(workflow.get_graph().draw_mermaid_png())

In [ ]:
# Invoke

initial_state = {'a': 1, 'b': 2, 'c': 3}
final_state = workflow.invoke(initial_state)

print(final_state)